# PDelta3 / GDN2 on arnir0/Tiny-LLM

This notebook checks whether the recurrent attention ideas that worked best in SmolLM2 also transfer to **arnir0/Tiny-LLM**.

Tiny-LLM has only **one Llama block**, so literal cross-layer value routing (CLVR) cannot exist. We therefore test:

1. `conv4_pdelta_f96` — current bounded-state control;
2. `conv4_channel_decay_f96` — stronger content-dependent channel decay;
3. `conv4_gdn2_f96` — independent erase/write GDN2 recurrence;
4. `conv4_gdn2_inputroute_f96` — a **one-layer value-routing proxy** that routes current pre-convolution V through the CLVR projection/gate. This is explicitly **not** true CLVR.

The host Transformer attention remains the baseline. Architecture selection uses validation only, while the held-out test set is evaluated once at the end with paired bootstrap confidence intervals.

After training, the notebook also reloads the validation-selected winner, runs deterministic prompt comparisons against original Tiny-LLM, exports a Hugging Face research-model package, builds a model card from the actual run, and uploads it with a token read securely from Colab Secrets.


In [ ]:
import os, sys, subprocess, tempfile
from pathlib import Path

assert subprocess.run(["nvidia-smi"], check=False).returncode == 0, "Enable a GPU runtime in Colab."
REPO = Path(tempfile.mkdtemp(prefix="TinyCeNN-tinyllm-"))
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6", "datasets", "huggingface_hub",
    "safetensors", "pandas", "matplotlib"
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repository:", REPO)
subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True)


In [ ]:
from datetime import datetime, timezone

PROFILE = "balanced"          # quick | balanced | strong
CURRICULUM = "256,512,1024"
VALIDATION_CONTEXTS = "256,512,1024"
TEST_CONTEXTS = "256,512,1024"
SEED = 2026

# Hugging Face publication settings.
# Add a write token in Colab: Secrets (key icon) -> HF_TOKEN.
HF_REPO_ID = "vtava/Tiny-LLM-PDelta3-GDN2-InputRoute"
HF_PRIVATE = False
UPLOAD_TO_HF = True

PROMPT_MAX_NEW_TOKENS = 64
PROMPTS = [
    "The future of small language models is",
    "Artificial intelligence can help scientists by",
    "A good software architecture should",
    "The key idea behind efficient language models is",
    "Once upon a time, a small robot",
]

RESULT_ROOT = Path("/content/TinyCeNN-tinyllm-results")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = RESULT_ROOT / f"{PROFILE}-{stamp}"
print("Output:", OUTPUT_DIR)
print("HF target:", HF_REPO_ID if UPLOAD_TO_HF else "upload disabled")


In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(
    "arnir0/Tiny-LLM",
    revision="b784a70a5e6908c9148820a245d60a3347279868",
)
print({
    "layers": cfg.num_hidden_layers,
    "hidden_size": cfg.hidden_size,
    "heads": cfg.num_attention_heads,
    "kv_heads": cfg.num_key_value_heads,
    "head_dim": cfg.hidden_size // cfg.num_attention_heads,
    "max_context": cfg.max_position_embeddings,
})
assert cfg.num_hidden_layers == 1
print("True CLVR is unavailable because Tiny-LLM has no preceding Transformer layer.")


In [ ]:
cmd = [
    sys.executable,
    str(REPO / "scripts" / "benchmark_tiny_llm_pdelta3_colab.py"),
    "--profile", PROFILE,
    "--curriculum", CURRICULUM,
    "--validation-contexts", VALIDATION_CONTEXTS,
    "--test-contexts", TEST_CONTEXTS,
    "--seed", str(SEED),
    "--output-dir", str(OUTPUT_DIR),
]
print(" ".join(cmd), flush=True)
process = subprocess.Popen(
    cmd, cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)


In [ ]:
import json
import pandas as pd
from IPython.display import display

validation = pd.read_csv(OUTPUT_DIR / "validation_context_summary.csv")
test = pd.read_csv(OUTPUT_DIR / "test_summary.csv")
diag = pd.read_csv(OUTPUT_DIR / "candidate_diagnostics.csv")
selection = json.loads((OUTPUT_DIR / "selection.json").read_text())
report = json.loads((OUTPUT_DIR / "tiny_llm_pdelta3_report.json").read_text())

print("SELECTION")
print(json.dumps(selection, indent=2))
print("\nVALIDATION")
display(validation.sort_values(["context", "delta_nll"]))
print("\nHELD-OUT TEST")
display(test.sort_values(["context", "delta_nll"]))
print("\nDIAGNOSTICS")
display(diag.sort_values("prefill_ms"))
strict = test[test["verdict"] == "strict_quality_win"]
print("\nSTRICT TRANSFORMER QUALITY WIN:", "YES" if len(strict) else "NO")
if len(strict):
    display(strict)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
for name, part in test.groupby("candidate"):
    part = part.sort_values("context")
    plt.plot(part["context"], part["delta_nll"], marker="o", label=name)
plt.axhline(0.0, linewidth=1)
plt.axhline(0.02, linewidth=1, linestyle="--")
plt.xlabel("Context length")
plt.ylabel("Candidate - Transformer NLL")
plt.title("Tiny-LLM: recurrent replacement quality gap")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(9, 5))
for name, part in test.groupby("candidate"):
    part = part.sort_values("context")
    plt.plot(part["context"], 100 * part["state_vs_transformer_fp16"], marker="o", label=name)
plt.xlabel("Context length")
plt.ylabel("Persistent state / Transformer FP16 KV (%)")
plt.title("Tiny-LLM persistent-state scaling")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


## Reload the selected model and test real prompts

This section uses the **validation-selected quality winner**. The test set is not used to choose the architecture.

Generation is deterministic greedy decoding and uses `use_cache=False`, because the research attention wrapper is evaluated as a full causal prefix. The same prompts and decoding rule are used for original Tiny-LLM and the PDelta3 replacement.


In [ ]:
import contextlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from scripts import benchmark_tiny_llm_pdelta3 as tinybench

BASE_MODEL = "arnir0/Tiny-LLM"
BASE_REVISION = "b784a70a5e6908c9148820a245d60a3347279868"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

winner = selection["quality_winner"]
spec = next(s for s in tinybench.candidate_specs() if s["name"] == winner)
checkpoint_path = OUTPUT_DIR / "checkpoints" / f"{winner}.pt"
assert checkpoint_path.exists(), checkpoint_path

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
original_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, torch_dtype=torch.float32
).to(DEVICE).eval()
adapted_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, torch_dtype=torch.float32
).to(DEVICE).eval()

payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
winner_core = tinybench.make_core(spec, adapted_model, DEVICE)
winner_core.load_state_dict(payload["state_dict"], strict=True)
winner_core.eval()

print("Loaded winner:", winner)
print("Spec:", spec)
print("Checkpoint:", checkpoint_path)
print("Persistent state bytes:", winner_core.recurrent_state_bytes())


In [ ]:
@torch.no_grad()
def greedy_generate(model, prompt, max_new_tokens=64, core=None, spec=None):
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    max_context = int(model.config.max_position_embeddings)
    max_new_tokens = min(max_new_tokens, max_context - ids.shape[1])
    if max_new_tokens <= 0:
        raise ValueError("Prompt already fills Tiny-LLM's context window.")

    manager = (
        tinybench.replace_attention(model, core, spec)
        if core is not None else contextlib.nullcontext()
    )
    with manager:
        for _ in range(max_new_tokens):
            logits = model(input_ids=ids, use_cache=False, return_dict=True).logits[:, -1, :]
            next_token = logits.argmax(dim=-1, keepdim=True)
            ids = torch.cat([ids, next_token], dim=1)
            if tokenizer.eos_token_id is not None and int(next_token.item()) == tokenizer.eos_token_id:
                break
    return tokenizer.decode(ids[0], skip_special_tokens=True)

prompt_rows = []
for prompt in PROMPTS:
    original_text = greedy_generate(original_model, prompt, PROMPT_MAX_NEW_TOKENS)
    pdelta3_text = greedy_generate(
        adapted_model, prompt, PROMPT_MAX_NEW_TOKENS,
        core=winner_core, spec=spec,
    )
    prompt_rows.append({
        "prompt": prompt,
        "original_tiny_llm": original_text,
        "pdelta3_selected": pdelta3_text,
    })

prompt_examples = pd.DataFrame(prompt_rows)
display(prompt_examples)
(OUTPUT_DIR / "prompt_examples.json").write_text(
    json.dumps(prompt_rows, indent=2, ensure_ascii=False), encoding="utf-8"
)
prompt_examples.to_csv(OUTPUT_DIR / "prompt_examples.csv", index=False)
print("Saved prompt examples to", OUTPUT_DIR)


## Build a Hugging Face research-model package

The base language model remains `arnir0/Tiny-LLM`. The selected recurrent core is saved as Safetensors, together with pinned architecture metadata, an inference script, benchmark evidence, real prompt examples, and a generated model card. This avoids pretending the custom recurrent attention is an ordinary unmodified `LlamaForCausalLM` checkpoint.


In [ ]:
import shutil
from safetensors.torch import save_file

HF_DIR = OUTPUT_DIR / "huggingface_model"
HF_DIR.mkdir(parents=True, exist_ok=True)

safe_state = {k: v.detach().cpu().contiguous() for k, v in payload["state_dict"].items()}
save_file(safe_state, str(HF_DIR / "pdelta3_core.safetensors"))

pdelta3_config = {
    "format": "TinyCeNN-PDelta3-attention-replacement-v1",
    "base_model_name_or_path": BASE_MODEL,
    "base_model_revision": BASE_REVISION,
    "selected_candidate": winner,
    "variant": spec["variant"],
    "route_mode": spec["route_mode"],
    "feature_dim": 96,
    "conv_kernel": 4,
    "state_dtype": "fp16",
    "tiny_llm_num_hidden_layers": int(adapted_model.config.num_hidden_layers),
    "true_clvr": False,
    "note": "Tiny-LLM has one Transformer block. InputRoute is a one-layer routing proxy, not true CLVR.",
}
(HF_DIR / "pdelta3_config.json").write_text(json.dumps(pdelta3_config, indent=2), encoding="utf-8")

for name in [
    "selection.json", "tiny_llm_pdelta3_report.json",
    "validation_context_summary.csv", "test_summary.csv",
    "candidate_diagnostics.csv", "training_history.csv",
    "prompt_examples.json", "prompt_examples.csv",
]:
    source = OUTPUT_DIR / name
    if source.exists():
        shutil.copy2(source, HF_DIR / name)

(HF_DIR / "requirements.txt").write_text(
    "transformers==4.57.6\ntorch\nsafetensors\nhuggingface_hub\n"
    "git+https://github.com/vtavakkoli/TinyCeNN-LM.git\n",
    encoding="utf-8",
)

inference_py = '''#!/usr/bin/env python3
import argparse, contextlib, json
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

from tinycenn_lm.pdelta3_frontier import FrontierPDelta3Layer


def rotate_half(x):
    a, b = x.chunk(2, dim=-1)
    return torch.cat((-b, a), dim=-1)


def project_qkv(attention, hidden, position_embeddings, num_heads, num_kv_heads):
    b, t, _ = hidden.shape
    q = attention.q_proj(hidden).view(b, t, num_heads, -1).transpose(1, 2)
    k = attention.k_proj(hidden).view(b, t, num_kv_heads, -1).transpose(1, 2)
    v = attention.v_proj(hidden).view(b, t, num_kv_heads, -1).transpose(1, 2)
    cos, sin = (x.unsqueeze(1) for x in position_embeddings)
    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin
    return q.float(), k.float(), v.float()


class TinyLLMReplacement(torch.nn.Module):
    def __init__(self, original, core, route_mode, num_heads, num_kv_heads):
        super().__init__()
        self.original = original
        self.core = core
        self.route_mode = route_mode
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads

    def forward(self, hidden_states, position_embeddings=None, attention_mask=None, **kwargs):
        if kwargs.get("use_cache", False):
            raise ValueError("Use use_cache=False for this research wrapper.")
        q, k, v = project_qkv(
            self.original, hidden_states, position_embeddings,
            self.num_heads, self.num_kv_heads,
        )
        route = v if self.route_mode == "current_v" else None
        output = self.core(q, k, v, routed_v=route)
        flat = output.transpose(1, 2).reshape(hidden_states.shape)
        return self.original.o_proj(flat.to(hidden_states.dtype)), None


@contextlib.contextmanager
def replace_attention(model, core, route_mode):
    original = model.model.layers[0].self_attn
    model.model.layers[0].self_attn = TinyLLMReplacement(
        original, core, route_mode,
        model.config.num_attention_heads,
        model.config.num_key_value_heads,
    )
    try:
        yield
    finally:
        model.model.layers[0].self_attn = original


def load_pdelta3(repo_id, device):
    config_path = hf_hub_download(repo_id, "pdelta3_config.json")
    weights_path = hf_hub_download(repo_id, "pdelta3_core.safetensors")
    cfg = json.loads(Path(config_path).read_text())
    model = AutoModelForCausalLM.from_pretrained(
        cfg["base_model_name_or_path"], revision=cfg["base_model_revision"],
        torch_dtype=torch.float32,
    ).to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(
        cfg["base_model_name_or_path"], revision=cfg["base_model_revision"]
    )
    core = FrontierPDelta3Layer(
        model.config.num_attention_heads,
        model.config.num_key_value_heads,
        model.config.hidden_size // model.config.num_attention_heads,
        feature_dim=cfg["feature_dim"],
        variant=cfg["variant"],
        chunk_size=32,
        conv_kernel=cfg["conv_kernel"],
        state_dtype=cfg["state_dtype"],
    ).to(device)
    core.load_state_dict(load_file(weights_path, device="cpu"), strict=True)
    core.eval()
    return model, tokenizer, core, cfg


@torch.no_grad()
def greedy(model, tokenizer, prompt, device, max_new_tokens=64, manager=None):
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"].to(device)
    max_new_tokens = min(max_new_tokens, model.config.max_position_embeddings - ids.shape[1])
    ctx = manager if manager is not None else contextlib.nullcontext()
    with ctx:
        for _ in range(max_new_tokens):
            logits = model(input_ids=ids, use_cache=False, return_dict=True).logits[:, -1]
            nxt = logits.argmax(dim=-1, keepdim=True)
            ids = torch.cat([ids, nxt], dim=1)
            if tokenizer.eos_token_id is not None and int(nxt.item()) == tokenizer.eos_token_id:
                break
    return tokenizer.decode(ids[0], skip_special_tokens=True)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--repo-id", required=True)
    p.add_argument("--prompt", default="The future of small language models is")
    p.add_argument("--max-new-tokens", type=int, default=64)
    p.add_argument("--compare-original", action="store_true")
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, tokenizer, core, cfg = load_pdelta3(args.repo_id, device)

    if args.compare_original:
        print("ORIGINAL TINY-LLM")
        print(greedy(model, tokenizer, args.prompt, device, args.max_new_tokens))
        print()

    print("PDELTA3")
    print(greedy(
        model, tokenizer, args.prompt, device, args.max_new_tokens,
        manager=replace_attention(model, core, cfg["route_mode"]),
    ))


if __name__ == "__main__":
    main()
'''
(HF_DIR / "inference.py").write_text(inference_py, encoding="utf-8")

license_text = '''MIT License

Copyright (c) 2026 Vahid Tavakkoli

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND.
'''
(HF_DIR / "LICENSE").write_text(license_text, encoding="utf-8")

print("Prepared HF export:", HF_DIR)
print(sorted(p.name for p in HF_DIR.iterdir()))


In [ ]:
winner_test = test[test["candidate"] == winner].sort_values("context").copy()

metric_lines = []
for _, row in winner_test.iterrows():
    metric_lines.append(
        f"| {int(row['context'])} | {row['transformer_ppl']:.4f} | "
        f"{row['candidate_ppl']:.4f} | {row['delta_nll']:+.6f} | "
        f"{(row['ppl_ratio']-1)*100:+.2f}% | "
        f"{row['state_vs_transformer_fp16']*100:.2f}% | "
        f"[{row['ci95_low']:+.6f}, {row['ci95_high']:+.6f}] |"
    )

example_sections = []
for item in prompt_rows:
    example_sections.append(
        "### Prompt\n\n```text\n" + item["prompt"] + "\n```\n\n"
        "**Original Tiny-LLM**\n\n```text\n" + item["original_tiny_llm"] + "\n```\n\n"
        f"**{winner}**\n\n```text\n" + item["pdelta3_selected"] + "\n```"
    )

readme = f'''---
license: mit
base_model: arnir0/Tiny-LLM
datasets:
- HuggingFaceFW/fineweb-edu
tags:
- tiny-llm
- pdelta3
- gdn2
- recurrent-attention
- bounded-state
- tinycenn
- research
---

# Tiny-LLM + PDelta3 / {winner}

This repository contains the **validation-selected recurrent attention replacement**
from the TinyCeNN Tiny-LLM compatibility experiment.

**Selected candidate:** `{winner}`

**Base model:** `arnir0/Tiny-LLM` at revision `{BASE_REVISION}`.

Tiny-LLM contains only one Transformer block, so this model cannot implement true
cross-layer value routing. The selected `InputRoute` configuration is a one-layer
proxy: it routes the current pre-convolution value representation through the
CLVR-style projection/gate. It should not be described as true CLVR.

## What changed?

The original Tiny-LLM attention is replaced at inference by a Conv4 recurrent
PDelta3/GDN2-style layer with bounded persistent state. The base embeddings, MLP,
normalization, output projection, and LM head remain from the original model.

The selected recurrent core is distributed in `pdelta3_core.safetensors`.
The base model weights are intentionally not duplicated; `inference.py` downloads
the pinned original Tiny-LLM revision and inserts this trained core.

## Held-out comparison with original Tiny-LLM

Architecture selection used validation data only. The held-out test documents were
not used to choose the winning candidate.

| Context | Original PPL | PDelta3 PPL | ΔNLL | PPL change | PDelta3 state / original FP16 KV | 95% CI ΔNLL |
|---:|---:|---:|---:|---:|---:|---:|
{chr(10).join(metric_lines)}

Lower PPL/NLL is better. A negative ΔNLL favors PDelta3. Positive ΔNLL means the
original Transformer attention remains better on held-out language-model quality.

## Prompt examples

The following examples were generated after training with deterministic greedy
decoding and the same prompt/max-token budget for both models.

{chr(10).join(example_sections)}

## Usage

```bash
pip install -r requirements.txt
python inference.py \
  --repo-id {HF_REPO_ID} \
  --compare-original \
  --prompt "The future of small language models is" \
  --max-new-tokens 64
```

## Files

- `pdelta3_core.safetensors` — selected recurrent-attention core.
- `pdelta3_config.json` — pinned base model/revision and architecture settings.
- `inference.py` — reconstructs the model and compares it to original Tiny-LLM.
- `test_summary.csv` — held-out evaluation with paired bootstrap intervals.
- `validation_context_summary.csv` — validation-only architecture selection.
- `candidate_diagnostics.csv` — timing/state/gate diagnostics.
- `training_history.csv` — progressive adaptation history.
- `prompt_examples.json` / `.csv` — deterministic generation comparison.
- `tiny_llm_pdelta3_report.json` and `selection.json` — complete experiment metadata.

## Training protocol

The recurrent replacement was progressively adapted at contexts `{CURRICULUM}`.
Selection used validation contexts `{VALIDATION_CONTEXTS}`.
Final held-out tests used `{TEST_CONTEXTS}`.

## Important limitations

- Tiny-LLM has only one Transformer block; true CLVR is structurally unavailable.
- The InputRoute variant is a routing proxy, not cross-layer CLVR.
- The supplied comparison generation uses `use_cache=False`.
- The original model remains better whenever held-out ΔNLL is positive.
- Reference PyTorch timing is not an optimized-kernel claim.
- This run uses one experiment seed.

## Reproducibility

Source and Colab: https://github.com/vtavakkoli/TinyCeNN-LM

Base model: https://huggingface.co/arnir0/Tiny-LLM

License: MIT.
'''
(HF_DIR / "README.md").write_text(readme, encoding="utf-8")
print((HF_DIR / "README.md").read_text()[:4000])


## Upload to Hugging Face

Create a Hugging Face **write** token and add it in Colab under **Secrets** with the name `HF_TOKEN`. The token is read at runtime and is never printed or written into exported files. If the secret is missing, the cell opens Hugging Face's secure interactive notebook login.


In [ ]:
from huggingface_hub import HfApi, notebook_login

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    return None

if UPLOAD_TO_HF:
    hf_token = get_hf_token()
    if hf_token is None:
        print("HF_TOKEN not found in environment/Colab Secrets. Opening secure Hugging Face login...")
        notebook_login()
        api = HfApi()
    else:
        api = HfApi(token=hf_token)

    api.create_repo(
        repo_id=HF_REPO_ID,
        repo_type="model",
        private=HF_PRIVATE,
        exist_ok=True,
    )
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(HF_DIR),
        commit_message=f"Upload {winner} from TinyCeNN Tiny-LLM compatibility benchmark",
    )
    print("Uploaded model package:")
    print(f"https://huggingface.co/{HF_REPO_ID}")
else:
    print("UPLOAD_TO_HF=False; package was created locally only:", HF_DIR)


In [ ]:
import shutil
from google.colab import files

hf_archive = shutil.make_archive(
    str(OUTPUT_DIR / "huggingface_model"),
    "zip",
    root_dir=HF_DIR,
)
experiment_archive = shutil.make_archive(
    str(OUTPUT_DIR),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Downloading Hugging Face model package:", hf_archive)
files.download(hf_archive)
print("Downloading complete experiment:", experiment_archive)
files.download(experiment_archive)
